Importing Cells and Set Up

In [37]:
import os
import cv2
import time
import torch
import numpy as np
from ultralytics import YOLO

Masking of the Safety Zone

In [38]:
def draw_mask_safely(frame, mask, color, alpha=0.5):
    """Safely draw a mask on a frame with validation"""
    try:
        # Validate mask
        if mask is None or len(mask) < 3:  # Need at least 3 points for a polygon
            return frame

        # Convert to correct format
        mask_points = np.array(mask, dtype=np.int32)
        if mask_points.shape[0] < 3:  # Double-check after conversion
            return frame

        # Create overlay
        overlay = frame.copy()
        cv2.fillPoly(overlay, [mask_points], color)
        return cv2.addWeighted(frame, 1-alpha, overlay, alpha, 0)

    except Exception as e:
        print(f"Warning: Could not draw mask: {e}")
        return frame

Proximity Detection Class 

In [39]:
import subprocess
import tempfile
from PIL import Image

class ProximityDetector:
    def __init__(self):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        # Load models
        self.walkable_model = YOLO('walkable_model.pt')
        self.object_model = YOLO('yolov8n-seg.pt')

        self.walkable_model.to(self.device)
        self.object_model.to(self.device)

        # Define immediate proximity zone (area right in front)
        self.proximity_height = 0.4  # Bottom 40% of frame
        self.proximity_width = 0.4   # Center 40% of frame width

        # Store the last results for printing
        self.walkable_results = None
        self.object_results = None
    
    def get_direction_from_vlm(self, frame):
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            Image.fromarray(frame).save(tmp.name)
            image_path = tmp.name

        prompt = "A robot is navigating this space. There's an obstacle ahead. Based on the environment, what is the best direction to move: left, right, forward, or stop? Only respond with one word: left, right, forward, or stop."

        try:
            result = subprocess.run(
                ['ollama', 'run', 'llava-phi3', '--image', image_path],
                input=prompt,
                text=True,
                capture_output=True,
                timeout=10
            )
            response = result.stdout.strip().lower()
            for word in ['left', 'right', 'forward', 'stop']:
                if word in response:
                    return word
            return "forward"  # fallback
        except Exception as e:
            print(f"Error getting VLM direction: {e}")
            return "forward"
        
    def print_model_stats(self):
        if self.walkable_results:
            result = self.walkable_results[0]
            img_shape = f"{result.orig_shape[0]}x{result.orig_shape[1]}"
            det_count = len(result.masks) if result.masks else 0

            print(f"0: {img_shape} {det_count} walkable-zone, {result.speed['inference']:.1f}ms")
            print(f"Speed: {result.speed['preprocess']:.1f}ms preprocess, "
                f"{result.speed['inference']:.1f}ms inference, "
                f"{result.speed['postprocess']:.1f}ms postprocess "
                f"per image at shape {tuple(result.boxes.orig_shape)}")
            print()

        if self.object_results:
            result = self.object_results
            img_shape = f"{result.orig_shape[0]}x{result.orig_shape[1]}"
            det_count = len(result.boxes) if result.boxes else 0

            print(f"0: {img_shape} {det_count} objects, {result.speed['inference']:.1f}ms")
            print(f"Speed: {result.speed['preprocess']:.1f}ms preprocess, "
                f"{result.speed['inference']:.1f}ms inference, "
                f"{result.speed['postprocess']:.1f}ms postprocess "
                f"per image at shape {tuple(result.boxes.orig_shape)}")
            print()


Proximity Checking Method

In [41]:
def is_in_proximity(self, box, frame_shape):
    height, width = frame_shape[:2]
    x1, y1, x2, y2 = box

    # Calculate box center
    box_center_x = (x1 + x2) / 2
    box_bottom_y = y2

    # Define zone: center 1/3 width, bottom 1/3 height
    zone_left = width / 3
    zone_right = 2 * width / 3
    zone_top = 2 * height / 3

    in_horizontal = zone_left < box_center_x < zone_right
    in_vertical = box_bottom_y > zone_top

    if in_horizontal and in_vertical:
        return True, 1  # You can later replace 1 with a better proximity score
    return False, 0


# Attach method to class
ProximityDetector.is_in_proximity = is_in_proximity

Frame Processing + Warning Message

In [42]:
def process_frame(self, frame):
    height, width = frame.shape[:2]
    processed = frame.copy()

    # 1. Detect walkable zones
    self.walkable_results = self.walkable_model.predict(frame, show=False)
    if self.walkable_results[0].masks is not None:
        for mask in self.walkable_results[0].masks.xy:
            # Safely draw walkable zones
            processed = draw_mask_safely(
                processed,
                mask,
                color=(0, 255, 0),  # Green for walkable zones
                alpha=0.3
            )
            # Add outline if mask is valid
            if len(mask) >= 3:
                cv2.polylines(processed, [np.int32(mask)], True, (0, 255, 0), 2)

    # 2. Detect and track objects
    self.object_results = self.object_model.track(frame, persist=True)[0]

    # Track closest obstacle
    closest_distance = 0
    has_close_obstacle = False

    if self.object_results.boxes is not None and self.object_results.masks is not None:
        boxes = self.object_results.boxes.xyxy.cpu().numpy()
        classes = self.object_results.boxes.cls.cpu().numpy()
        masks = self.object_results.masks.xy
        track_ids = self.object_results.boxes.id.cpu().numpy() if self.object_results.boxes.id is not None else None

        for i, (box, cls, mask) in enumerate(zip(boxes, classes, masks)):
            # Safely draw objects
            processed = draw_mask_safely(
                processed,
                mask,
                color=(255, 165, 0),  # Orange for objects
                alpha=0.5
            )

            # Add tracking ID and label
            if track_ids is not None and len(mask) > 0:
                track_id = int(track_ids[i])
                class_name = self.object_model.names[int(cls)]
                label = f"ID {track_id} - {class_name}"
                x_min, y_min = mask.min(axis=0)
                cv2.putText(processed, label,
                          (int(x_min), int(y_min) - 10),
                          cv2.FONT_HERSHEY_SIMPLEX,
                          0.6,
                          (255, 255, 255),
                          2)

            # Check proximity for specific obstacle classes
            class_name = self.object_model.names[int(cls)]
            if class_name in ['person', 'bicycle', 'car']:
                in_proximity, distance = self.is_in_proximity(box, frame.shape)
                if in_proximity and distance > 0.3:  # Object significantly in front
                    has_close_obstacle = True
                    if distance > closest_distance:
                        closest_distance = distance

                    # Draw red box around closest obstacle
                    cv2.rectangle(processed,
                        (int(box[0]), int(box[1])),
                        (int(box[2]), int(box[3])),
                        (0, 0, 255), 2)

        # Add warning text if obstacle detected
        if has_close_obstacle:
            direction = self.get_direction_from_vlm(frame)
            warning_text = f"Obstacle ahead, please move {direction}"

            text_size = cv2.getTextSize(warning_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
            text_x = (width - text_size[0]) // 2
            cv2.putText(processed, warning_text,
                      (text_x, height - 50),
                      cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                      (0, 0, 255), 3)

    # Print YOLO processing details
    self.print_model_stats()

    return processed

# Attach method to class
ProximityDetector.process_frame = process_frame

Model Statistics

In [43]:
def print_model_stats(self):
    """Print the statistics from the YOLO models"""
    if self.walkable_results:
        result = self.walkable_results[0]
        img_shape = f"{result.orig_shape[0]}x{result.orig_shape[1]}"
        det_count = 0
        if result.masks is not None:
            det_count = len(result.masks)

        print(f"0: {img_shape} {det_count} walkable-zone, {result.speed['inference']:.1f}ms")
        print(f"Speed: {result.speed['preprocess']:.1f}ms preprocess, "
              f"{result.speed['inference']:.1f}ms inference, "
              f"{result.speed['postprocess']:.1f}ms postprocess "
              f"per image at shape {tuple(result.boxes.orig_shape)}")
        print()  # Empty line

    if self.object_results:
        result = self.object_results
        img_shape = f"{result.orig_shape[0]}x{result.orig_shape[1]}"
        det_count = 0
        if result.boxes is not None:
            det_count = len(result.boxes)

        print(f"0: {img_shape} {det_count} persons, {result.speed['inference']:.1f}ms")
        print(f"Speed: {result.speed['preprocess']:.1f}ms preprocess, "
              f"{result.speed['inference']:.1f}ms inference, "
              f"{result.speed['postprocess']:.1f}ms postprocess "
              f"per image at shape {tuple(result.boxes.orig_shape)}")
        print()  # Empty line

# Attach method to class
ProximityDetector.print_model_stats = print_model_stats

Video Processing 

In [44]:
def process_video(input_path, output_path):
    import cv2
    import time
    import os

    os.makedirs("videos", exist_ok=True)

    detector = ProximityDetector()

    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"Error: Could not open video file {input_path}")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        processed_frame = detector.process_frame(frame)
        out.write(processed_frame)

        frame_count += 1

    cap.release()
    out.release()
    print(f"✅ Video processed and saved to {output_path}")


Main

In [46]:
# Main execution
input_video = "videos/clip_107_to_159.mp4"
output_video = 'videos/processed_video2.mp4'

# Process the video
process_video(input_video, output_video)

Using device: cpu

0: 384x640 1 walkable-zone, 39.5ms
Speed: 1.3ms preprocess, 39.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 43.3ms
Speed: 1.1ms preprocess, 43.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0: 2160x3840 1 walkable-zone, 39.5ms
Speed: 1.3ms preprocess, 39.5ms inference, 0.7ms postprocess per image at shape (2160, 3840)

0: 2160x3840 1 persons, 43.3ms
Speed: 1.1ms preprocess, 43.3ms inference, 1.0ms postprocess per image at shape (2160, 3840)


0: 384x640 1 walkable-zone, 51.2ms
Speed: 1.7ms preprocess, 51.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 45.0ms
Speed: 1.0ms preprocess, 45.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0: 2160x3840 1 walkable-zone, 51.2ms
Speed: 1.7ms preprocess, 51.2ms inference, 0.8ms postprocess per image at shape (2160, 3840)

0: 2160x3840 1 persons, 45.0ms
Speed: 1.0ms preprocess, 45.0ms inference, 1.0ms pos